In [1]:
import os
import sys
import numpy as np
import pandas as pd
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
from sklearn.model_selection import train_test_split

import scanpy as sc
import anndata

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
)

from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
import evaluate

sys.path.append("..")
from lib_vqvae.vqvae import VQVAE
from lib_vqvae.dataset import *

#### Load the pre-trained cell tokenizer

In [2]:
import warnings
warnings.filterwarnings("ignore")
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--train_batch_size", type=int, default=64)
parser.add_argument("--eval_batch_size", type=int, default=2000)
parser.add_argument("--encoder_hidden_dim", type=list, default=[1600,1024,800])
parser.add_argument("--decoder_hidden_dim", type=list, default=[800,1024,1600])
parser.add_argument("--codebook_dim", type=int, default=8)
parser.add_argument("--n", type=int, default=32)
parser.add_argument("--n_codebooks", type=int, default=256)
parser.add_argument("--gamma", type=float, default=5)
parser.add_argument("--beta", type=float, default=10)
parser.add_argument('--weight_decay', type=float, default=5e-4)
parser.add_argument("--lr", type=float, default=1e-4) # 1e-3
parser.add_argument('--max_iter', type=int, default=20)
parser.add_argument('--seed', type=int, default=2024)
parser.add_argument('--gpu', type=int, default=0)
parser.add_argument("--dropout_rate", type=float, default=0.2)
parser.add_argument('--mode', type=str, default="vqvae")
parser.add_argument('--hvg', type=int, default=5000)

args = parser.parse_args([])

# -- device
if torch.cuda.is_available():
    args.device='cuda'
    torch.cuda.set_device(args.gpu)
else:
    args.device='cpu' 
print(args.device)
    
data_dim = 5000
vae = VQVAE(data_dim, args.n, args.codebook_dim, args.n_codebooks, args.encoder_hidden_dim, args.decoder_hidden_dim, args.beta, args.gamma, args.dropout_rate).to(args.device)

vae.load_state_dict(torch.load('../model/cell_tokenzier/model.pth')['model'])
vae.eval() 

cuda


VQVAE(
  (encoder): Encoder(
    (network): Sequential(
      (0): Linear(in_features=5000, out_features=1600, bias=True)
      (1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
      (2): PReLU(num_parameters=1)
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=1600, out_features=1024, bias=True)
      (5): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (6): PReLU(num_parameters=1)
      (7): Dropout(p=0.2, inplace=False)
      (8): Linear(in_features=1024, out_features=800, bias=True)
      (9): LayerNorm((800,), eps=1e-05, elementwise_affine=True)
      (10): PReLU(num_parameters=1)
      (11): Dropout(p=0.2, inplace=False)
      (12): Linear(in_features=800, out_features=256, bias=True)
    )
  )
  (vector_quantization): VectorQuantizer(
    (embedding): Embedding(256, 8)
  )
  (decoder): Decoder(
    (network): Sequential(
      (0): Linear(in_features=256, out_features=800, bias=True)
      (1): LayerNorm((800,), eps=1e-05, elementwise

#### Prepare Model and Tokenizer
Now, we will be adding 258 new tokens as well as replace the existing pad, bos and eos tokens of the model.

In [3]:
# Define new tokens
new_tokens = [f'<code_{i}>' for i in range(256)] + ['<cell_token_start>', '<cell_token_end>']

In [4]:
model_name = "your path to the LLM backbone"  # load the base LLM backbone
tokenizer_path = "your path to LLM's tokenizer"  # load the LLM toknizer

tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_path,
    pad_token="<|pad|>",
    eos_token="<|endsentence|>",
    additional_special_tokens=new_tokens,
)

In [5]:
len(tokenizer)

151925

#### Load the pre-trained CellTok model

In [6]:
from peft import PeftModel

inference_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
)
inference_model.resize_token_embeddings(len(tokenizer))

inference_model = PeftModel.from_pretrained(inference_model, "your path to CellTok-base")

inference_model.to("cuda")
inference_model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

PeftModel(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): lora.Embedding(
          (base_layer): Embedding(151925, 3584)
          (lora_dropout): ModuleDict(
            (default): Identity()
          )
          (lora_A): ModuleDict()
          (lora_B): ModuleDict()
          (lora_embedding_A): ParameterDict(  (default): Parameter containing: [torch.cuda.FloatTensor of size 64x151925 (cuda:0)])
          (lora_embedding_B): ParameterDict(  (default): Parameter containing: [torch.cuda.FloatTensor of size 3584x64 (cuda:0)])
          (lora_magnitude_vector): ModuleDict()
        )
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
     

#### Interactive Q&A

In [7]:
cell_code = "<cell_token_start><code_251><code_159><code_200><code_231><code_90><code_151><code_250><code_189>\
<code_211><code_160><code_2><code_96><code_124><code_67><code_69><code_95><code_215><code_55><code_94><code_219><code_59>\
<code_188><code_2><code_117><code_224><code_89><code_245><code_231><code_3><code_211><code_219><code_78><cell_token_end>"

In [11]:
dialogue_history = []  

cell_prefix = f"{cell_code}, tell me some information about this cell."
dialogue_history.append(cell_prefix)

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ['exit', 'quit']:
        break

    full_context = "\n".join(dialogue_history + [f"<|user|> {user_input}\n<|assistant|>"])

    batch = tokenizer(full_context, return_tensors="pt")
    batch = {k: v.to("cuda") for k, v in batch.items()}

    output_tokens = inference_model.generate(
        **batch,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.5,
        top_p=0.95,
        top_k=50,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    decoded = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    if '<|assistant|>' in decoded:
        assistant_reply = decoded.split('<|assistant|>')[-1].split('<|im_end|>')[0].strip()
    else:
        assistant_reply = decoded.strip()

    print(f"CellTok: {assistant_reply}")
    dialogue_history.append(f"<|user|> {user_input}")
    dialogue_history.append(f"<|assistant|> {assistant_reply}")

You:  hello


CellTok: Hi there! Is there anything specific you need help with?


You:  <cell_token_start><code_251><code_159><code_200><code_231><code_90><code_151><code_250><code_189>\ <code_211><code_160><code_2><code_96><code_124><code_67><code_69><code_95><code_215><code_55><code_94><code_219><code_59>\ <code_188><code_2><code_117><code_224><code_89><code_245><code_231><code_3><code_211><code_219><code_78><cell_token_end>, tell me some information about this cell.


CellTok: This cell appears to be a neuron from the cerebral cortex of a 72-year-old female human, identified using the 10x 3' v3 assay, and is from a individual with small vessel disease and self-reported European ethnicity.


You:  What is small vessel disease?


CellTok: Small vessel disease (SVD) is a term used to describe damage to the small blood vessels in the brain. It is a common cause of dementia, particularly in older adults. The exact cause of SVD is not fully understood, but it is thought to be related to both small vessel changes such as endothelial dysfunction and microvascular dementia. There are also several risk factors for SVD, including hypertension, hyperlipidemia, diabetes, and smoking. It is important to note that the cerebral cortex is the part of the brain responsible for processing information from all senses, and for producing coordinated muscle movements.


You:  I get it, thank you!


CellTok: You're welcome! If you have any more questions or need further clarification, don't hesitate to let me know.


You:  exit
